In [14]:
# DS4400 HW 3
# Eunchae Hong
# Problem 2: Gradient Descent for Logistic Regression
# Take 3 values of the learning rate and report the cross-entropy loss objective after 10, 50, and 100 iterations
# At 100 iterations, report the accuracy, precision, recall, and F1 score for the 3 learning rates, 
# and compare with the metrics given by the package on the training and testing sets.

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score
)

# load the spam database in and implement the same train_test_split
spam_data = pd.read_csv("spambase/spambase.data", header = None)
X = spam_data.iloc[:, :-1].values
y = spam_data.iloc[:, -1].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.25, random_state = 5
)

In [ ]:
# define functions needed for the problem
# defining the sigmoid function
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

# defining the cross entropy loss function
def cross_entropy_loss(y, y_prob):
    N = len(y)
    # clip to avoid log(0)
    y_prob = np.clip(y_prob, 1e-15, 1 - 1e-15)
    return (-1 / N) * np.sum(y * np.log(y_prob) + (1 - y) * np.log(1 - y_prob))

# gradient descent function from HW2 with changes for Logistic Regression
def gradient_descent(X, y, alpha, iterations):
    ''' function that implements a gradient descent for training for logistic regression (changed from hw2)'''
    N, d = X.shape
    theta = np.zeros(d)
    iteration_vals = [10, 50, 100]
    losses = {}

    for i in range(1, iterations + 1):
        y_pred = sigmoid(X @ theta)
        gradient = (1 / N) * X.T @ (y_pred - y)
        theta = theta - alpha * gradient

        if i in iteration_vals:
            loss = cross_entropy_loss(y, sigmoid(X @ theta))
            losses[i] = loss

    return theta, losses

In [16]:
# set learning rates
learning_rates = [0.1, 0.05, 0.005]

# print out the cross entropy loss with the different learning rates at 10, 50, and 100 iterations
results = {}
for alpha in learning_rates:
    theta, losses = gradient_descent(X_train, y_train, alpha, 100)
    results[alpha] = theta
    print(f"Learning Rate: {alpha}")
    print(f"Cross Entropy Loss at 10: {round(losses[10], 3)}\nCross Entropy Loss at 50: {round(losses[50], 3)}\nCross Entropy Loss at 100: {round(losses[100], 3)}")

Learning Rate: 0.1
Cross Entropy Loss at 10: 19.228
Cross Entropy Loss at 50: 13.095
Cross Entropy Loss at 100: 20.057
Learning Rate: 0.05
Cross Entropy Loss at 10: 17.745
Cross Entropy Loss at 50: 12.767
Cross Entropy Loss at 100: 12.025
Learning Rate: 0.005
Cross Entropy Loss at 10: 7.306
Cross Entropy Loss at 50: 11.523
Cross Entropy Loss at 100: 11.162


In [18]:
# create a function to report the metrics
def report_metrics(label, y_true, y_pred):
    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    print(f"{label}")
    print(f"Accuracy: {round(accuracy, 3)} Precision: {round(precision, 3)}"
          f" Recall: {round(recall, 3)} F1 Score: {round(f1, 3)}")

# print the metrics from my model
print("Metrics From My Model on Training and Testing Sets")
for alpha in learning_rates:
    theta = results[alpha]
    y_pred_train = (sigmoid(X_train @ theta) >= 0.5).astype(int)
    y_pred_test  = (sigmoid(X_test  @ theta) >= 0.5).astype(int)
    report_metrics(f"Learning Rate({alpha}) Training Metrics:", y_train, y_pred_train)
    report_metrics(f"Learning Rate({alpha}) Testing Metrics:",  y_test,  y_pred_test)

print("-" * 60)

# using the sklearn logistic regression model to print metrics and compare
print("Metrics from the Packagae on Training and Testing Sets")
package_model = LogisticRegression(max_iter = 10000, solver = "lbfgs", random_state = 5)
package_model.fit(X_train, y_train)
report_metrics("Package Train Metrics:", y_train, package_model.predict(X_train))
report_metrics("Package Test Metrics:",  y_test,  package_model.predict(X_test))

Metrics From My Model on Training and Testing Sets
Learning Rate(0.1) Training Metrics:
Accuracy: 0.415 Precision: 0.404 Recall: 0.999 F1 Score: 0.576
Learning Rate(0.1) Testing Metrics:
Accuracy: 0.41 Precision: 0.395 Recall: 1.0 F1 Score: 0.566
Learning Rate(0.05) Training Metrics:
Accuracy: 0.508 Precision: 0.441 Recall: 0.89 F1 Score: 0.59
Learning Rate(0.05) Testing Metrics:
Accuracy: 0.511 Precision: 0.434 Recall: 0.889 F1 Score: 0.583
Learning Rate(0.005) Training Metrics:
Accuracy: 0.614 Precision: 0.825 Recall: 0.034 F1 Score: 0.066
Learning Rate(0.005) Testing Metrics:
Accuracy: 0.625 Precision: 0.867 Recall: 0.029 F1 Score: 0.057
------------------------------------------------------------
Metrics from the Packagae on Training and Testing Sets
Package Train Metrics:
Accuracy: 0.93 Precision: 0.929 Recall: 0.891 F1 Score: 0.91
Package Test Metrics:
Accuracy: 0.929 Precision: 0.917 Recall: 0.896 F1 Score: 0.906
